# Stage 6.2 – Independent Blinded Judge: Mistral 7B Instruct v0.3

This notebook is one of three independent LLM judges. It evaluates the 45 frozen reports without seeing Condition A, B or C labels.

Rules preserved in this notebook:

- the same question-specific PMC passages are supplied for all three reports;
- visible citations and evidence-source lists are removed from judge-facing reports;
- report order is shuffled deterministically for this judge and saved separately;
- generation uses temperature 0;
- raw outputs and parsing failures are retained;
- the judge does not generate or edit the research reports.


## 1. Select a free Colab GPU and install packages

Choose **Runtime → Change runtime type → T4 GPU** before running. The model is loaded in 4-bit form to fit the free T4 memory.


In [ ]:
!pip -q install "transformers==4.49.0" "accelerate==1.3.0" "bitsandbytes==0.45.3" "sentencepiece==0.2.0"


## 2. Upload and verify the frozen evaluation input

Upload `Adarsh_Konderu_Stage_6_Frozen_Evaluation_Input.zip` from the Stage 6.2 pack.


In [ ]:
from google.colab import files
from pathlib import Path
import csv
import hashlib
import importlib.metadata
import json
import platform
import random
import re
import shutil
import time
import zipfile

EXPECTED_INPUT_SHA256 = "d47dadcef96fa2f492fb05d6876ca0f3bd851c4b3583b090a6ba39557517513e"
uploaded = files.upload()
input_name = next(iter(uploaded))
input_bytes = uploaded[input_name]
actual_input_hash = hashlib.sha256(input_bytes).hexdigest()
print("Expected SHA-256:", EXPECTED_INPUT_SHA256)
print("Uploaded SHA-256:", actual_input_hash)
assert actual_input_hash == EXPECTED_INPUT_SHA256, "Upload the frozen input supplied with this Stage 6.2 pack."

work_dir = Path("/content/stage_6_2_mistral_work")
if work_dir.exists():
    shutil.rmtree(work_dir)
work_dir.mkdir(parents=True)

with zipfile.ZipFile(input_name) as archive:
    archive.extractall(work_dir)

input_dir = work_dir / "stage_6_frozen_evaluation_input"
cases = [json.loads(line) for line in (input_dir / "evaluation_cases.jsonl").read_text().splitlines() if line.strip()]
assert len(cases) == 15
assert sum(len(case["reports"]) for case in cases) == 45
print("Verified 15 questions and 45 frozen reports.")


## 3. Record the fixed judge and rubric

All three notebooks use the same rubric. The complete prompt is hashed and stored in the final trace.


In [ ]:
JUDGE_ID = "mistral"
JUDGE_NAME = "Mistral 7B Instruct v0.3"
MODEL_REPO = "mistralai/Mistral-7B-Instruct-v0.3"
MODEL_REVISION = "c170c708c41dac9275d15a8fff4eca08d52bab71"
MODEL_LICENSE = "Apache-2.0"
TRUST_REMOTE_CODE = False
RANDOMISATION_NAMESPACE = "STAGE_6_2_BALANCED_BLIND_ORDER_V2_2026-09-03"
RUBRIC = "You are an independent evaluator of research reports about AI in healthcare.\nUse only the question and the supplied reference passages. Do not use outside knowledge.\nThe three candidate reports are anonymous. Do not guess how they were produced.\n\nScore every candidate from 1 to 5 on each criterion:\n- factual_support: 1 = substantially contradicted or unsupported; 3 = mixed support with notable unsupported details; 5 = claims are consistently supported by the references and not contradicted.\n- relevance: 1 = mostly off-topic; 3 = answers some important parts; 5 = directly answers all parts of the question.\n- coherence: 1 = disorganised or difficult to follow; 3 = understandable with weaknesses; 5 = clear, logically organised and internally consistent.\n- completeness: 1 = major required aspects absent; 3 = partial coverage; 5 = covers the important aspects supported by the supplied references.\n- responsible_framing: 1 = unsafe or unjustifiably certain healthcare claims; 3 = some uncertainty or safeguards but important omissions; 5 = limitations, uncertainty and appropriate safeguards are clearly communicated.\n\nApply the same standard to all candidates. A report does not need visible citations because citation markers were deliberately removed for blinding.\nReturn JSON only, using exactly this structure:\n{\"reports\":[{\"label\":\"Report 1\",\"factual_support\":1,\"relevance\":1,\"coherence\":1,\"completeness\":1,\"responsible_framing\":1,\"brief_reason\":\"one concise sentence\"},{\"label\":\"Report 2\",\"factual_support\":1,\"relevance\":1,\"coherence\":1,\"completeness\":1,\"responsible_framing\":1,\"brief_reason\":\"one concise sentence\"},{\"label\":\"Report 3\",\"factual_support\":1,\"relevance\":1,\"coherence\":1,\"completeness\":1,\"responsible_framing\":1,\"brief_reason\":\"one concise sentence\"}]}"
RUBRIC_SHA256 = hashlib.sha256(RUBRIC.encode()).hexdigest()

print("Judge:", JUDGE_NAME)
print("Model:", MODEL_REPO)
print("Revision:", MODEL_REVISION)
print("Rubric SHA-256:", RUBRIC_SHA256)


## 4. Load the judge model

Four-bit NF4 quantisation reduces memory use. Quantisation is applied identically as an inference setting and the original pinned model repository is retained in the trace.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed

assert torch.cuda.is_available(), "A GPU is required. Select a T4 GPU in Runtime settings."
set_seed(42)

quantisation = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_REPO,
    revision=MODEL_REVISION,
    trust_remote_code=TRUST_REMOTE_CODE,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

judge_model = AutoModelForCausalLM.from_pretrained(
    MODEL_REPO,
    revision=MODEL_REVISION,
    quantization_config=quantisation,
    device_map="auto",
    trust_remote_code=TRUST_REMOTE_CODE,
    low_cpu_mem_usage=True,
)
judge_model.eval()
print("Loaded", JUDGE_NAME, "on", judge_model.device)


## 5. Build anonymous prompts and validate JSON

The condition mapping is retained for later analysis but is never included in the text sent to the judge model.


In [ ]:
CRITERIA = ["factual_support", "relevance", "coherence", "completeness", "responsible_framing"]
EXPECTED_LABELS = ["Report 1", "Report 2", "Report 3"]
CASE_INDEX = {case["question_id"]: index for index, case in enumerate(cases)}


def anonymous_reports(case):
    """Return a balanced deterministic order and its sealed mapping.

    Each three-question block uses all three cyclic positions of a shuffled
    base order. Across 15 questions, each condition therefore occurs exactly
    five times in each judge-facing position.
    """
    case_index = CASE_INDEX[case["question_id"]]
    block, rotation = divmod(case_index, 3)
    seed_text = f"{RANDOMISATION_NAMESPACE}|{JUDGE_ID}|block-{block}"
    seed = int(hashlib.sha256(seed_text.encode()).hexdigest()[:16], 16)
    condition_order = ["A", "B", "C"]
    random.Random(seed).shuffle(condition_order)
    condition_order = condition_order[rotation:] + condition_order[:rotation]
    report_by_condition = {report["condition"]: report for report in case["reports"]}
    shuffled = [report_by_condition[condition] for condition in condition_order]
    labelled = []
    mapping = []
    for number, report in enumerate(shuffled, start=1):
        label = f"Report {number}"
        labelled.append((label, report["judge_view_text"]))
        mapping.append({"label": label, "condition": report["condition"], "judge_view_sha256": report["judge_view_sha256"]})
    return labelled, mapping


def build_prompt(case, labelled_reports):
    references = "\n\n".join(
        f"Reference passage {number} — {item['title']} ({item['year']}):\n{item['text']}"
        for number, item in enumerate(case["references"], start=1)
    )
    candidates = "\n\n".join(f"### {label}\n{text}" for label, text in labelled_reports)
    return f"{RUBRIC}\n\nRESEARCH QUESTION:\n{case['question']}\n\nREFERENCE PASSAGES:\n{references}\n\nCANDIDATE REPORTS:\n{candidates}"


def extract_json(raw_text):
    """Extract and strictly validate the requested JSON object."""
    start, end = raw_text.find("{"), raw_text.rfind("}")
    if start < 0 or end <= start:
        raise ValueError("No complete JSON object found")
    data = json.loads(raw_text[start:end + 1])
    reports = data.get("reports")
    if not isinstance(reports, list) or len(reports) != 3:
        raise ValueError("Expected exactly three report results")
    by_label = {item.get("label"): item for item in reports}
    if set(by_label) != set(EXPECTED_LABELS):
        raise ValueError("Missing or duplicated anonymous report labels")
    for label in EXPECTED_LABELS:
        item = by_label[label]
        for criterion in CRITERIA:
            score = item.get(criterion)
            if not isinstance(score, int) or isinstance(score, bool) or not 1 <= score <= 5:
                raise ValueError(f"Invalid {criterion} score for {label}")
        if not isinstance(item.get("brief_reason"), str) or not item["brief_reason"].strip():
            raise ValueError(f"Missing brief reason for {label}")
    return {"reports": [by_label[label] for label in EXPECTED_LABELS]}


## 6. Define deterministic model generation

Only newly generated tokens are decoded. If the first response is not valid JSON, one deterministic repair request is made and both raw responses are saved.


In [ ]:
def generate_text(prompt, max_new_tokens=700):
    messages = [
        {"role": "system", "content": "Follow the evaluation instructions exactly and return JSON only."},
        {"role": "user", "content": prompt},
    ]
    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(judge_model.device)
    input_length = encoded["input_ids"].shape[-1]
    with torch.inference_mode():
        generated = judge_model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(generated[0][input_length:], skip_special_tokens=True).strip()


def evaluate_case(case):
    labelled, mapping = anonymous_reports(case)
    prompt = build_prompt(case, labelled)
    started = time.time()
    first_raw = generate_text(prompt)
    repair_raw = None
    parse_status = "first_response_valid"
    parse_error = None
    try:
        parsed = extract_json(first_raw)
    except Exception as first_error:
        repair_prompt = (
            "Convert the response below into the exact JSON schema required by the original rubric. "
            "Preserve any scores already present. If a field was omitted, evaluate it using the original material. "
            "Return JSON only.\n\nORIGINAL TASK:\n" + prompt + "\n\nRESPONSE TO REPAIR:\n" + first_raw
        )
        repair_raw = generate_text(repair_prompt)
        try:
            parsed = extract_json(repair_raw)
            parse_status = "repair_response_valid"
        except Exception as repair_error:
            parsed = None
            parse_status = "invalid_after_one_repair"
            parse_error = f"first={type(first_error).__name__}: {first_error}; repair={type(repair_error).__name__}: {repair_error}"
    return {
        "question_id": case["question_id"],
        "prompt_sha256": hashlib.sha256(prompt.encode()).hexdigest(),
        "mapping": mapping,
        "first_raw_response": first_raw,
        "repair_raw_response": repair_raw,
        "parse_status": parse_status,
        "parse_error": parse_error,
        "parsed": parsed,
        "duration_seconds": round(time.time() - started, 3),
    }


## 7. Run the 15-question evaluation

This makes one evaluation call per question. A checkpoint is written after every question. Depending on the judge model, the run may take approximately 15–40 minutes on a free T4 GPU.


In [ ]:
results = []
checkpoint_path = work_dir / "mistral_judge_checkpoint.json"

for number, case in enumerate(cases, start=1):
    result = evaluate_case(case)
    results.append(result)
    checkpoint_path.write_text(json.dumps(results, indent=2, ensure_ascii=False))
    print(f"{number}/15 {case['question_id']}: {result['parse_status']} ({result['duration_seconds']:.1f} s)")

valid_count = sum(item["parsed"] is not None for item in results)
print("Valid questions:", valid_count, "/ 15")
print("The evidence package will retain any invalid result as missing; no score is invented.")


## 8. Decode labels only after judging and save evidence

This cell creates 45 score rows if all 15 responses were valid. It also saves the raw responses, condition mapping, trace, protocol and checksums.


In [ ]:
output_dir = Path("/content/stage_6_2_mistral_judge_evidence")
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True)

score_rows = []
mapping_rows = []
raw_rows = []

for result in results:
    condition_by_label = {item["label"]: item["condition"] for item in result["mapping"]}
    for item in result["mapping"]:
        mapping_rows.append({"question_id": result["question_id"], **item})
    raw_rows.append({
        "question_id": result["question_id"],
        "prompt_sha256": result["prompt_sha256"],
        "parse_status": result["parse_status"],
        "parse_error": result["parse_error"],
        "duration_seconds": result["duration_seconds"],
        "first_raw_response": result["first_raw_response"],
        "repair_raw_response": result["repair_raw_response"],
    })
    if result["parsed"] is None:
        continue
    for scored in result["parsed"]["reports"]:
        scores = [scored[name] for name in CRITERIA]
        score_rows.append({
            "judge_id": JUDGE_ID,
            "question_id": result["question_id"],
            "anonymous_label": scored["label"],
            "condition": condition_by_label[scored["label"]],
            **{name: scored[name] for name in CRITERIA},
            "criterion_mean": sum(scores) / len(scores),
            "brief_reason": scored["brief_reason"],
            "parse_status": result["parse_status"],
        })

def write_csv(path, rows):
    if not rows:
        path.write_text("")
        return
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

write_csv(output_dir / "judge_scores.csv", score_rows)
write_csv(output_dir / "anonymous_condition_mapping.csv", mapping_rows)
(output_dir / "raw_judge_outputs.jsonl").write_text(
    "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in raw_rows)
)
shutil.copy(input_dir / "STAGE_6_EVALUATION_PROTOCOL.md", output_dir / "STAGE_6_EVALUATION_PROTOCOL.md")

trace = {
    "stage": "Stage 6.2 independent blinded LLM judge",
    "judge_id": JUDGE_ID,
    "judge_name": JUDGE_NAME,
    "model_repo": MODEL_REPO,
    "model_revision": MODEL_REVISION,
    "model_license": MODEL_LICENSE,
    "quantisation": "bitsandbytes NF4 4-bit, double quantisation, float16 compute",
    "temperature": 0,
    "random_seed": 42,
    "randomisation_namespace": RANDOMISATION_NAMESPACE,
    "position_balance": "Each condition appears five times in each anonymous report position.",
    "rubric_sha256": RUBRIC_SHA256,
    "input_archive_sha256": actual_input_hash,
    "question_count": 15,
    "valid_question_count": sum(item["parsed"] is not None for item in results),
    "score_row_count": len(score_rows),
    "repair_attempt_count": sum(item["repair_raw_response"] is not None for item in results),
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "cuda_device": torch.cuda.get_device_name(0),
    "package_versions": {name: importlib.metadata.version(name) for name in ["transformers", "accelerate", "bitsandbytes"]},
}
(output_dir / "judge_trace.json").write_text(json.dumps(trace, indent=2))

checksums = {
    path.name: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in sorted(output_dir.iterdir()) if path.is_file()
}
(output_dir / "checksums_sha256.json").write_text(json.dumps(checksums, indent=2))

archive_path = shutil.make_archive(
    f"/content/Adarsh_Konderu_Stage_6_2_Mistral_Judge_Evidence",
    "zip",
    output_dir.parent,
    output_dir.name,
)
print("Score rows:", len(score_rows), "/ 45")
print("Saved:", archive_path)
files.download(archive_path)
